<a href="https://colab.research.google.com/github/shruthilakshmi008-BA/ba-automation-suite/blob/main/04.%20Invoice-Extractor/04_Invoice_Extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-genai pandas

In [3]:
import os
import re
import json
import pandas as pd

from google import genai
from google.colab import files, userdata


# Load Gemini API key securely from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. "
        "Please add it under Colab → Secrets."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini API key loaded successfully.")
print("Gemini client initialized successfully.")

Gemini API key loaded successfully.
Gemini client initialized successfully.


In [4]:
uploaded = files.upload()

input_file = list(uploaded.keys())[0]

print(f"Input file uploaded: {input_file}")

Saving raw_invoices.txt to raw_invoices.txt
Input file uploaded: raw_invoices.txt


In [5]:
##Read invoice file + Regex rules - This cell reads the uploaded file and defines the deterministic extraction layer.
with open(input_file, "r", encoding="utf-8") as f:
    raw_invoices = f.read().split("---")


FIELD_PATTERNS = {
    "invoice_number": r"Invoice\s*#?:?\s*([A-Za-z0-9\-]+)",
    "vendor": r"Vendor:\s*([^\-\n]+)",
    "date": r"Date:\s*([\d/\-]+)",
    "total": r"Total:\s*\$?([\d,]+\.\d{2})",
}


def extract_with_rules(raw_text: str) -> dict:
    """Deterministic Regex Pattern Extraction."""

    result = {}

    for field, pattern in FIELD_PATTERNS.items():

        match = re.search(
            pattern,
            raw_text,
            re.IGNORECASE
        )

        result[field] = (
            match.group(1).strip()
            if match
            else None
        )

    return result


print(f"Number of invoice sections found: {len(raw_invoices)}")

Number of invoice sections found: 3


In [8]:
##Gemini AI extraction for missing fields
def extract_with_ai(
    raw_text: str,
    missing_fields: list
) -> dict:

    try:

        print(
            f"  [Gemini AI] Resolving missing fields: "
            f"{missing_fields}"
        )

        prompt = f"""
You are an expert Document Processing System.

Extract ONLY the requested missing metadata fields
from the invoice text below.

Invoice Raw Text:
{raw_text}

Missing Fields:
{missing_fields}

Return ONLY a valid JSON object.

Allowed fields:

- invoice_number: string
- vendor: company name
- date: date string
- total: numeric string such as 1,234.56

Rules:

1. Return only the requested fields.
2. Do not invent information.
3. If a field cannot be found, return null.
4. Do not include currency symbols in total.
5. Do not include markdown.
6. Do not include ```json.
7. Do not provide explanations.
"""

        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        raw_content = response.text.strip()

        # Remove markdown fences if Gemini returns them
        raw_content = re.sub(
            r"^```json\s*|\s*```$",
            "",
            raw_content,
            flags=re.IGNORECASE
        ).strip()

        parsed_json = json.loads(raw_content)

        return {
            field: parsed_json.get(field)
            for field in missing_fields
        }

    except Exception as e:

        print(
            f"  [Gemini Warning] Extraction failed: {e}"
        )

        print(
            "  [Audit Flag] Fields marked as NEEDS_REVIEW."
        )

        return {
            field: "NEEDS_REVIEW"
            for field in missing_fields
        }

In [9]:
##Process all invoices - Main Processing Engine
rows = []

for i, raw_text in enumerate(
    raw_invoices,
    start=1
):

    raw_text = raw_text.strip()

    if not raw_text:
        continue

    source_id = f"SRC-{i:03d}"

    print(
        f"\nProcessing invoice: {source_id}"
    )

    # -----------------------------------
    # Layer 1: Regex extraction
    # -----------------------------------

    extracted = extract_with_rules(raw_text)

    missing = [
        field
        for field, value in extracted.items()
        if value is None
    ]

    extraction_method = "Deterministic Regex"

    # -----------------------------------
    # Layer 2: Gemini extraction
    # -----------------------------------

    if missing:

        extraction_method = (
            "Hybrid (Regex + Gemini/Review)"
        )

        ai_result = extract_with_ai(
            raw_text,
            missing
        )

        extracted.update(ai_result)

    # -----------------------------------
    # Standardized output
    # -----------------------------------

    ordered_row = {
        "source_id": source_id,
        "invoice_number": extracted.get(
            "invoice_number"
        ),
        "vendor": extracted.get(
            "vendor"
        ),
        "date": extracted.get(
            "date"
        ),
        "total": extracted.get(
            "total"
        ),
        "extraction_method": extraction_method
    }

    rows.append(ordered_row)


print(
    f"\nSuccessfully processed "
    f"{len(rows)} invoices."
)


Processing invoice: SRC-001

Processing invoice: SRC-002
  [Gemini AI] Resolving missing fields: ['vendor', 'date', 'total']

Processing invoice: SRC-003

Successfully processed 3 invoices.


In [10]:
## Ouptut file generation
df = pd.DataFrame(rows)

print("Invoice extraction results:")
print("=" * 80)

display(df)

output_file = "invoice_extraction_output.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(
    f"Output file saved successfully: "
    f"{output_file}"
)
files.download(output_file)

Invoice extraction results:


,source_id,invoice_number,vendor,date,total,extraction_method
0,SRC-001,INV-2001,Acme Supplies Co,09/12/2026,"1,245.00",Deterministic Regex
1,SRC-002,INV-2002,Global Traders,09/13/2026,980.50,Hybrid (Regex + Gemini/Review)
2,SRC-003,INV-2003,Bright Logistics,09/14/2026,"2,310.75",Deterministic Regex


Output file saved successfully: invoice_extraction_output.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>